In [1]:
import os
import sqlite3

basedir = os.path.abspath(os.path.dirname("__file__"))


class Config(object):
    DEBUG = False
    TESTING = False
    CSRF_ENABLED = True
    SECRET_KEY = "this-really-needs-to-be-changed"
    DATABASE_PATH = os.path.join(basedir, "jobs.db")

    def get_db(self):
        return sqlite3.connect(self.DATABASE_PATH)


class ProductionConfig(Config):
    DEBUG = False
    # Production should use a real DB via DATABASE_PATH env var or override


class StagingConfig(Config):
    DEVELOPMENT = True
    DEBUG = True


class DevelopmentConfig(Config):
    DEVELOPMENT = True
    DEBUG = True
    DATABASE_PATH = os.path.join(basedir, "dev.db")


class TestingConfig(Config):
    TESTING = True
    DATABASE_PATH = ":memory:"

In [ ]:
import requests
from bs4 import BeautifulSoup
from scrape_unpaid import scrape_unpaid
from scrape_paid import scrape_paid
import sqlite3
import os

db_path = os.path.join(os.path.abspath(os.path.dirname("__file__")), "jobs.db")

page_number_limit = 5000
last_known_result = None


def build_url(min_date, max_date, page_number):
    return (
        "https://www.jobindex.dk/jobsoegning?"
        + "maxdate="
        + max_date
        + "&mindate="
        + min_date
        + "&page="
        + str(page_number)
        + "&archive=1" # Inkluderer arkiverede jobopslag
    )


def scrape(max_date, min_date, page_number):
    last_known_date = max_date

    url = build_url(min_date, max_date, page_number)
    print("fetching data from url: {0}".format(url))
    page = requests.get(url)
    soup = BeautifulSoup(page.content, "html.parser")

    if soup.find("div", class_="PaidJob"): # Selve jobstillingen på jobindex
        last_known_date = (
            soup.find("div", class_="PaidJob").find("time").attrs["datetime"]
        )

    if soup.find("div", class_="jix-toolbar-top__company"):
        last_known_date = (
            soup.find("a", target="_blank").get_text()
        )

    if soup.find("span", class_="jix_robotjob--area"): # jobplaceringen
        location = soup.find("span", class_="jix_robotjob--area").get_text()

    ## do page request
    if len(page.content) > 0:
        conn = sqlite3.connect(db_path)
        try:
            ## fetch unpaid
            scrape_unpaid(soup, conn)

            ## fetch paid
            scrape_paid(soup, conn)

            conn.commit()
        finally:
            conn.close()

        return last_known_date

    return False


def format_date(date):
    return ("").join((date).split("-"))


def url_format_date(date):
    return ("").join(date.split("-"))


def run(page_number, min_date, max_date):
    # run batch of scrapes
    while page_number <= page_number_limit:

        last_known_result = scrape(
            page_number=page_number, max_date=max_date, min_date=min_date
        )

        # if the requests returns content
        if last_known_result != False:
            print("\n\n\n")
            print("last known date: {0}".format(last_known_result))
            page_number += 1
        else:
            print("shows over!")
            return False

    proper_date = url_format_date(last_known_result)
    print("about to start a new batch. last known date is: {0}".format(proper_date))
    run(page_number=1, min_date=min_date, max_date=proper_date)

In [5]:
db_path

'c:\\Users\\irisg\\Documents\\GitHub\\jobindex-scraper\\jobs.db'